# 03. Experimentación y Selección de Modelos

Con los datos limpios, enriquecidos y escalados, es hora de encontrar el algoritmo predictivo ganador.

### Instrucciones Generales:
1. **Validación:** No entrenes y midas sobre el mismo conjunto (sobreajuste). Recuerda haber dividido en Entrenamiento y Prueba antes.
2. **Entrenamiento Base:** Entrena los siguientes modelos base con tu set de Entrenamiento y compáralos usando RMSE (Error Cuadrático Medio):
   - `LinearRegression`
   - `SGDRegressor`
   - `DecisionTreeRegressor`
   - `RandomForestRegressor`
3. **Cross Validation (Validación Cruzada):** Para tener una métrica robusta, usa `cross_val_score` en el set de Entrenamiento para cada uno de los modelos anteriores.
4. **Ajuste Fino (Fine Tuning):** Toma el modelo ganador y busca sus mejores hiperparámetros. Utiliza un `GridSearchCV` explorando el número de estimadores (`n_estimators`), las características máximas (`max_features`), etc.
5. **Conclusión y Benchmark (IMPORTANTE):** Redacta una conclusión comparando los algoritmos. Explica por qué escogiste el modelo final y valida tu decisión calculando el RMSE sobre tu Set de Prueba que habías reservado. Documenta si alguno de tus modelos se sobreajusto o subajusto. Recuerda que el modelo final no puede tener esos problemas!


In [125]:
# Empieza importando los algoritmos desde Scikit-Learn
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
import pandas as pd
import numpy as np


In [126]:
# Cargamos los datos procesados
df_procesado = pd.read_csv("../data/processed/train_processed.csv")

print("X_train:", df_procesado.shape)


X_train: (13209, 13)


1. Escalado y separación de target

In [127]:
# Escalado de variables
# dividir el conjunto de datos en características (X) y variable objetivo (y)
X_train = df_procesado.drop("median_house_value", axis=1)
y_train = df_procesado["median_house_value"]

# Escalar las variables numéricas para mejorar el rendimiento de los modelos de machine learning
scaler = StandardScaler()
numerical_features = X_train.select_dtypes(include="number").columns
X_train[numerical_features] = scaler.fit_transform(X_train[numerical_features])

2. Entrenamiento

In [128]:
# Definimos los 4 modelos base
modelos = {
    "LinearRegression":       LinearRegression(),
    "SGDRegressor":           SGDRegressor(random_state=42),
    "DecisionTreeRegressor":  DecisionTreeRegressor(random_state=42),
    "RandomForestRegressor":  RandomForestRegressor(random_state=42)
}

# Entrenamos cada modelo y calculamos RMSE sobre el train
print("Cálculo de RMSE sobre modelos de entrenamiento:")
print("-"*47)

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    prediccion = modelo.predict(X_train)
    error_train = mean_squared_error(y_train, prediccion)
    rmse = np.sqrt(error_train)
    print(f"{nombre:30s}: {rmse:,.0f}")

Cálculo de RMSE sobre modelos de entrenamiento:
-----------------------------------------------
LinearRegression              : 67,626
SGDRegressor                  : 509,712
DecisionTreeRegressor         : 0
RandomForestRegressor         : 18,707


3. Cross Validation

In [131]:
# Cross Validation con 5 folds para cada modelo
print("RMSE con Cross Validation (5 folds):")
print("-"*36)

for nombre, modelo in modelos.items():
    scores = cross_val_score(
        modelo, X_train, y_train,
        scoring="neg_mean_squared_error",
        cv=5
    )
    rmse_cv = np.sqrt(-scores).mean()
    std_cv  = np.sqrt(-scores).std()
    print(f"{nombre:30s}: {rmse_cv:,.0f} ± {std_cv:,.0f}")

RMSE con Cross Validation (5 folds):
------------------------------------
LinearRegression              : 67,780 ± 2,494
SGDRegressor                  : 37,479,083 ± 74,532,763
DecisionTreeRegressor         : 70,888 ± 2,931
RandomForestRegressor         : 50,640 ± 2,799


El mejor modelo en mi caso el Random Forest REgressor, ya que en la predicción por RMSE y Cross Validation obtiene el menor valor.

4. Fine Tuning - Ajuste fino

In [137]:
# Parámetros para grid search en RandomForestRegressor
parametros = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20],
    "min_samples_split": [2, 5]
}

In [138]:
# grid search para RandomForestRegressor
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(),
    param_grid=parametros,
    cv=5,
    verbose=2,
)
grid_search.fit(X_train, y_train)
print("Mejores hiperparámetros:", grid_search.best_params_)
print(f"Mejor RMSE CV: ${np.sqrt(-grid_search.best_score_):,.0f}")

Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV] END max_depth=10, min_samples_split=2, n_estimators=100; total time=   4.2s
[CV] END max_depth=10, min_samples_split=2, n_estimators=100; total time=   4.0s
[CV] END max_depth=10, min_samples_split=2, n_estimators=100; total time=   4.2s
[CV] END max_depth=10, min_samples_split=2, n_estimators=100; total time=   4.6s
[CV] END max_depth=10, min_samples_split=2, n_estimators=100; total time=   4.4s
[CV] END max_depth=10, min_samples_split=2, n_estimators=200; total time=   8.2s
[CV] END max_depth=10, min_samples_split=2, n_estimators=200; total time=   9.7s
[CV] END max_depth=10, min_samples_split=2, n_estimators=200; total time=   8.5s
[CV] END max_depth=10, min_samples_split=2, n_estimators=200; total time=   8.5s
[CV] END max_depth=10, min_samples_split=2, n_estimators=200; total time=   8.9s
[CV] END max_depth=10, min_samples_split=5, n_estimators=100; total time=   4.1s
[CV] END max_depth=10, min_samples_split=5, n_est

C:\Users\anita\AppData\Local\Temp\ipykernel_4044\504439468.py:10: RuntimeWarning: invalid value encountered in sqrt
  print(f"Mejor RMSE CV: ${np.sqrt(-grid_search.best_score_):,.0f}")


### Benchmark y Conclusión Final
*(Escribe tus conclusiones de negocio aquí)*

In [ ]:
# Cargamos el set de validación
X_val = pd.read_csv("../data/interim/val_set.csv")
y_val = X_val.pop("median_house_value")

# Aplicamos el mismo pipeline de preprocesamiento
import sys
sys.path.append("../src/features")
from build_features import preprocess_pipeline

# Procesamos val igual que train
val_df = pd.read_csv("../data/interim/val_set.csv")
X_val_proc, y_val_proc = preprocess_pipeline(val_df)

# Evaluamos el mejor modelo
mejor_modelo = grid_search.best_estimator_
pred_val = mejor_modelo.predict(X_val_proc)
rmse_val = np.sqrt(mean_squared_error(y_val_proc, pred_val))

print(f"RMSE en validación: ${rmse_val:,.0f}")